# Product Clustering — EDA

Explore unique products to decide how to group them into 4-6 meta-categories.

In [1]:
#kernal kept on crashing, tried this to make it work on cpu but it didn't help > I'll try google colab
'''
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
'''

'\nimport os\nos.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\n\nfrom sentence_transformers import SentenceTransformer\nfrom sklearn.cluster import KMeans\nimport matplotlib.pyplot as plt\n\nembedder = SentenceTransformer(\'sentence-transformers/all-MiniLM-L6-v2\', device=\'cpu\')\n'

In [2]:
import sys
sys.path.append('../..')

import pandas as pd

df = pd.read_csv('../../datasets/merged_reviews.csv')
print(f'{len(df)} reviews loaded')

59743 reviews loaded


In [3]:


products = df[['product_id', 'product_title']].drop_duplicates(subset=['product_title']).reset_index(drop=True)
print(f'{len(products)} unique products (by cleaned product_title) out of {len(df)} reviews')
products.sample(10)

72 unique products (by cleaned product_title) out of 59743 reviews


,product_id,product_title
15,AVpjEN4jLJeJML43rpUe,Fire Kids Edition Tablet
49,AWYAV-i9Iwln0LfXqrUq,Echo Spot Pair Kit (Black)
67,AVzYlGkFvKc47QAVeZRI,All-New Fire HD 8 Tablet with Alexa
35,AVpfl8cLLJeJML43AE3S,Amazon Kindle Fire 5ft USB to Micro-USB Cable ...
5,AVqkIj9snnc1JgDc3khU,Amazon 5W USB Official OEM Charger and Power A...
64,AVzYlGj3vKc47QAVeZRH,"Amazon Fire HD 8 with Alexa (8"" HD Display Tab..."
12,AVphgVaX1cnluZ0-DR74,Certified Refurbished Amazon Fire TV (Previous...
50,AWFFfd9KIwln0LfXiOe0,Fire TV Stick Streaming Media Player Pair Kit
56,AVpfpK8KLJeJML43BCuD,Amazon Tap Smart Assistant Alexaenabled (black...
47,AVpfl8cLLJeJML43AE3S,Amazon Echo ‚Äì White


## Title length / word count
Sanity-check the text we're about to embed.

In [4]:
products['title_word_count'] = products['product_title'].astype(str).str.split().apply(len)
products['title_word_count'].describe()

count    72.000000
mean      7.486111
std       3.939661
min       1.000000
25%       4.000000
50%       7.000000
75%      10.000000
max      18.000000
Name: title_word_count, dtype: float64

## Most common words in product titles
Rough signal for what natural categories might exist before we cluster.

In [5]:
from collections import Counter
import re

words = Counter()
for title in products['product_title'].astype(str):
    words.update(re.findall(r'[a-zA-Z]{3,}', title.lower()))

words.most_common(30)

[('amazon', 41),
 ('fire', 30),
 ('kindle', 28),
 ('tablet', 14),
 ('with', 13),
 ('echo', 13),
 ('black', 11),
 ('generation', 11),
 ('new', 10),
 ('reader', 9),
 ('alexa', 9),
 ('usb', 9),
 ('all', 7),
 ('and', 7),
 ('power', 7),
 ('leather', 6),
 ('adapter', 6),
 ('cover', 5),
 ('charger', 5),
 ('white', 5),
 ('micro', 5),
 ('oasis', 4),
 ('certified', 4),
 ('refurbished', 4),
 ('edition', 4),
 ('cable', 4),
 ('charging', 3),
 ('for', 3),
 ('voyage', 3),
 ('case', 3)]

## Choosing k: elbow method on embeddings
Encode product titles once, then check inertia across a few candidate k values (4-6 per the brief) before committing in `01_baseline_KMeans.ipynb`.

**Note:** this cell loads a sentence-transformer model and can be slow on first run / CPU-only machines.

In [6]:
#!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
products = products.dropna(subset=['product_title']).reset_index(drop=True)
embeddings = embedder.encode(products['product_title'].astype(str).tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

: 

In [ ]:
inertias = []
candidate_ks = range(3, 8)
for k in candidate_ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(embeddings)
    inertias.append(km.inertia_)

plt.plot(list(candidate_ks), inertias, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow method for choosing k')
plt.show()

Pick the k where inertia stops dropping sharply (the 'elbow'), constrained to 4-6 per the project brief. Use that value as `n_clusters` in `01_baseline_KMeans.ipynb`.